In [2]:
pip install akshare

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 29.8 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 40.5 MB/s  0:00:00
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ---------------------- ----------------- 8.9/15.5 MB 42.6 MB/s eta 0:00:01
   ---------------------------------------- 15.5/15.5 MB 40.7 MB/s  0:00:00

   -- -------------------------------------  1/17 [webencodings]
   ---- -----------------------------------  2/17 [urllib3]
   ---- -----------------------------------  2/17 [urllib3]
   ------- --------------------------------  3/17 [soupsieve]
   ----------- ----------------------------  5/17 [mini-racer]
   ----------- ---------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import akshare as ak

def get_shebao_holdings(date_str="20260630"):
    """
    获取指定报告期的社保基金持仓
    date_str 格式: 'YYYYMMDD' (如 20260630 代表 2026年中报)
    """
    print(f"正在获取 {date_str} 报告期的社保基金持仓数据...")
    try:
        # 获取东方财富网-特定报告期的机构持仓明细
        # 机构类型包含：券商、基金、QFII、保险、社保等
        df_institution = ak.stock_report_disclosure_cninfo(market="深沪京", period=date_str)
        
        # 过滤出股东名称中包含“社保基金”的记录
        # 注：不同版本的接口字段名可能略有差异，通常包含 '股东名称' 或 '机构名称'
        shebao_df = df_institution[df_institution['十大流通股东'].str.contains('社保基金|全国社保', na=False, regex=True)]
        
        return shebao_df
    except Exception as e:
        print(f"数据获取失败: {e}")
        return pd.DataFrame()

# 示例：对比 2026 一季报 和 2026 年中报，找出连续增持个股
if __name__ == "__main__":
    df_q1 = get_shebao_holdings("20260331")
    df_q2 = get_shebao_holdings("20260630")
    
    if not df_q1.empty and not df_q2.empty:
        # 1. 聚合各股票的社保持仓总量
        q1_total = df_q1.groupby('股票代码')['持股数量(股)'].sum().reset_index(name='Q1_持股')
        q2_total = df_q2.groupby('股票代码')['持股数量(股)'].sum().reset_index(name='Q2_持股')
        
        # 2. 合并两个季度的数据
        merged_df = pd.merge(q1_total, q2_total, on='股票代码', how='inner')
        
        # 3. 筛选环比增持的股票
        increased_stocks = merged_df[merged_df['Q2_持股'] > merged_df['Q1_持股']]
        
        print("\n--- 满足社保基金环比增持的个股 ---")
        print(increased_stocks)


正在获取 20260331 报告期的社保基金持仓数据...
数据获取失败: module 'akshare' has no attribute 'stock_report_disclosure_cninfo'
正在获取 20260630 报告期的社保基金持仓数据...
数据获取失败: module 'akshare' has no attribute 'stock_report_disclosure_cninfo'


In [13]:
import akshare as ak
import pandas as pd

def get_ssf_holdings(report_date: str) -> pd.DataFrame | None:
    print(f"正在获取 {report_date} 报告期的社保基金持仓数据...")
    try:
        # symbol options: 基金持仓 / QFII持仓 / 社保持仓 / 券商持仓 / 保险持仓 / 信托持仓
        df = ak.stock_report_fund_hold(symbol="社保持仓", date=report_date)
        print(f"获取成功: {len(df)} 只股票")
        return df
    except Exception as e:
        print(f"数据获取失败: {e}")
        return None

results = {d: get_ssf_holdings(d) for d in ["20260930"]}


正在获取 20260930 报告期的社保基金持仓数据...
获取成功: 598 只股票


In [19]:
display(results["20260930"].groupby("持股变化").size().reset_index(name="数量"))

,持股变化,数量
0,不变,119
1,减仓,137
2,增仓,197
3,新进,145


In [27]:
display(results["20260930"][results["20260930"]["持股变化"].isin(["新进", "增仓","不变"])])

,序号,股票代码,股票简称,持有基金家数,持股总数,持股市值,持股变化,持股变动数值,持股变动比例
0,1,000680,山推股份,4,95215960,8.626566e+08,增仓,7322600,8.33
1,2,002318,久立特材,4,50713448,9.042208e+08,增仓,2912000,6.09
2,3,002444,巨星科技,4,57979960,1.556182e+09,增仓,3805157,7.02
3,4,000528,柳工,3,37541577,2.823127e+08,增仓,1163800,3.20
5,6,000700,模塑科技,3,40280997,4.946506e+08,增仓,33672697,509.55
...,...,...,...,...,...,...,...,...,...
593,594,688786,悦安新材,1,2071238,5.410074e+07,不变,0,0.00
594,595,688789,宏华数科,1,5547042,2.773521e+08,新进,5547042,NaN
595,596,688808,联讯仪器,1,268462,4.152114e+08,新进,268462,NaN
596,597,688813,泰金新能,1,370456,4.745171e+07,增仓,71050,23.73


In [29]:
df = results["20260930"]
display(
    df[df["持股变化"].isin(["新进", "增仓", "不变"])]
    .sort_values("持股市值", ascending=False).head(50)
)

,序号,股票代码,股票简称,持有基金家数,持股总数,持股市值,持股变化,持股变动数值,持股变动比例
233,234,002594,比亚迪,1,54704200,4.557407e+09,增仓,2963488,5.73
451,452,601899,紫金矿业,1,85798682,2.555943e+09,增仓,4798729,5.92
58,59,300628,亿联网络,2,50057410,1.976767e+09,增仓,8722230,21.10
95,96,601888,中国中免,2,33192098,1.707402e+09,增仓,7993184,31.72
88,89,601058,赛轮轮胎,2,127998380,1.694699e+09,增仓,51935000,68.28
275,276,300124,汇川技术,1,32600000,1.691940e+09,不变,0,0.00
2,3,002444,巨星科技,4,57979960,1.556182e+09,增仓,3805157,7.02
464,465,603087,甘李药业,1,18614962,1.440798e+09,增仓,4637500,33.18
32,33,002027,分众传媒,2,303986001,1.437854e+09,增仓,7166300,2.41
246,247,002714,牧原股份,1,31868817,1.352831e+09,新进,31868817,NaN
